In [1]:
import json
import random
from pathlib import Path
from collections import Counter

In [2]:
path1=Path('../datasets/review_pairs.json')
path2=Path('../datasets/reviews.json')

with path1.open('r',encoding='utf-8') as file:
    pairs=json.load(file)

with path2.open('r',encoding='utf-8') as file:
    reviews=json.load(file)

print(len(pairs),len(reviews))

3311 3287


In [3]:
def extract_boxed(text: str):
    start=text.rfind('\\boxed{')
    if start==-1:
        return None

    i=start+len('\\boxed{')
    depth=1
    while i<len(text) and depth>0:
        if text[i]=='{':
            depth+=1
        elif text[i]=='}':
            depth-=1
        i+=1

    if depth!=0:
        return None
    return text[start+len('\\boxed{'):i-1]

In [4]:
random.seed(0)

review_data=[]
refine_data=[]
cot_data=[]

for i in reviews:
    final_answer=extract_boxed(i['review'])

    # C: two chains + teacher review
    review_data.append({
        'id': i['id'],
        'type': i['type'],
        'question': i['problem'],
        'chains': [i['answer_1'],i['answer_2']],
        'review': i['review'],
        'final_answer': final_answer,
    })

    # B: two chains + one more correct chain (prefer one the model has not shown yet)
    shown=[i['answer_1'],i['answer_2']]
    unseen=[c for c in i['other_correct'] if c not in shown]
    refined=random.choice(unseen) if unseen else random.choice(i['other_correct'])

    refine_data.append({
        'id': i['id'],
        'type': i['type'],
        'question': i['problem'],
        'chains': shown,
        'refined': refined,
        'final_answer': extract_boxed(refined),
    })

    # conventional CoT: one correct chain, same problems
    cot_data.append({
        'id': i['id'],
        'type': i['type'],
        'question': i['problem'],
        'chains': [random.choice(i['other_correct'])],
        'final_answer': None,
    })
    cot_data[-1]['final_answer']=extract_boxed(cot_data[-1]['chains'][0])

print(len(review_data),len(refine_data),len(cot_data))

3287 3287 3287


In [5]:
Counter(i['type'] for i in review_data)

Counter({'correction': 1166, 'keep': 1122, 'agree': 999})

In [6]:
assert all(i['final_answer']!=None for i in review_data)
assert all(i['final_answer']!=None for i in refine_data)
assert all(i['final_answer']!=None for i in cot_data)
assert all(len(i['chains'])==2 for i in review_data)
assert all(len(i['chains'])==1 for i in cot_data)

In [7]:
review_data[0]

{'id': 'math-2021',
 'type': 'correction',
 'question': 'Expand $(2z^2 + 5z - 6)(3z^3 - 2z + 1)$.',
 'chains': ['\\begin{aligned} (2z^2 + 5z - 6)(3z^3 - 2z + 1) &= 2z^2(3z^3 - 2z + 1) + 5z(3z^3 - 2z + 1) - 6(3z^3 - 2z + 1) \\\\ &= 6z^5 - 4z^3 + 2z^2 + 15z^4 - 10z^2 + 5z - 18z^3 + 12z - 6 \\\\ &= 6z^5 + 15z^4 - 22z^3 - 10z^2 + 12z - 6 \\end{aligned} \\boxed{6z^5 + 15z^4 - 22z^3 - 10z^2 + 12z - 6}',
  'First, we need to expand the product \\((2z^2 + 5z - 6)(3z^3 - 2z + 1)\\). We will use the distributive property (also known as the FOIL method for binomials or the distributive property for polynomials). \\[ (2z^2 + 5z - 6)(3z^3 - 2z + 1) \\] We distribute each term in the first polynomial to each term in the second polynomial: \\[ 2z^2 \\cdot 3z^3 + 2z^2 \\cdot (-2z) + 2z^2 \\cdot 1 + 5z \\cdot 3z^3 + 5z \\cdot (-2z) + 5z \\cdot 1 - 6 \\cdot 3z^3 - 6 \\cdot (-2z) - 6 \\cdot 1 \\] Now, we compute each term: \\[ 2z^2 \\cdot 3z^3 = 6z^5 \\] \\[ 2z^2 \\cdot (-2z) = -4z^3 \\] \\[ 2z^2 \\cdot 

In [8]:
refine_data[0]

{'id': 'math-2021',
 'type': 'correction',
 'question': 'Expand $(2z^2 + 5z - 6)(3z^3 - 2z + 1)$.',
 'chains': ['\\begin{aligned} (2z^2 + 5z - 6)(3z^3 - 2z + 1) &= 2z^2(3z^3 - 2z + 1) + 5z(3z^3 - 2z + 1) - 6(3z^3 - 2z + 1) \\\\ &= 6z^5 - 4z^3 + 2z^2 + 15z^4 - 10z^2 + 5z - 18z^3 + 12z - 6 \\\\ &= 6z^5 + 15z^4 - 22z^3 - 10z^2 + 12z - 6 \\end{aligned} \\boxed{6z^5 + 15z^4 - 22z^3 - 10z^2 + 12z - 6}',
  'First, we need to expand the product \\((2z^2 + 5z - 6)(3z^3 - 2z + 1)\\). We will use the distributive property (also known as the FOIL method for binomials or the distributive property for polynomials). \\[ (2z^2 + 5z - 6)(3z^3 - 2z + 1) \\] We distribute each term in the first polynomial to each term in the second polynomial: \\[ 2z^2 \\cdot 3z^3 + 2z^2 \\cdot (-2z) + 2z^2 \\cdot 1 + 5z \\cdot 3z^3 + 5z \\cdot (-2z) + 5z \\cdot 1 - 6 \\cdot 3z^3 - 6 \\cdot (-2z) - 6 \\cdot 1 \\] Now, we compute each term: \\[ 2z^2 \\cdot 3z^3 = 6z^5 \\] \\[ 2z^2 \\cdot (-2z) = -4z^3 \\] \\[ 2z^2 \\cdot 

In [9]:
savepath1=Path('../datasets/sft_review.json')
savepath2=Path('../datasets/sft_refine.json')
savepath3=Path('../datasets/sft_cot.json')

with savepath1.open('w',encoding='utf-8') as file:
    json.dump(review_data,file,ensure_ascii=False,indent=2)

with savepath2.open('w',encoding='utf-8') as file:
    json.dump(refine_data,file,ensure_ascii=False,indent=2)

with savepath3.open('w',encoding='utf-8') as file:
    json.dump(cot_data,file,ensure_ascii=False,indent=2)

print("Saved all three")

Saved all three
